In [1]:
import os
import json
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv(".env")
api_key = os.getenv("AZURE_OPENAI_API_KEY")

In [2]:
# Function to generate and execute curl commands
def run_curl_request(model, prompt_text, temperature=0.7):
    # Create the curl command
    cmd = f"""curl -X POST "https://ai-giotesthub416362359489.openai.azure.com/openai/responses?api-version=2025-03-01-preview" \
-H "Content-Type: application/json" \
-H "api-key: {api_key}" \
-d '{{
  "model": "{model}",
  "input": "{prompt_text}",
  "temperature": {temperature},
  "truncation": "auto"
}}'"""
    
    # Execute the command and capture output
    import subprocess
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Parse the response
    if result.returncode == 0:
        import json
        try:
            return json.loads(result.stdout)
        except json.JSONDecodeError:
            return {"error": "Failed to parse JSON response", "raw_output": result.stdout}
    else:
        return {"error": result.stderr}

# Example usage
response = run_curl_request("computer-use-preview", "What is the capital of France?")
print(json.dumps(response, indent=2))

{
  "id": "resp_680beca0f4948190969d2d0740a84f5908435c3c1af64552",
  "object": "response",
  "created_at": 1745611936,
  "status": "completed",
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "max_output_tokens": null,
  "model": "computer-use-preview",
  "output": [
    {
      "id": "msg_680beca16a948190ae201a602ae0842a08435c3c1af64552",
      "type": "message",
      "status": "completed",
      "content": [
        {
          "type": "output_text",
          "annotations": [],
          "text": "The capital of France is Paris."
        }
      ],
      "role": "assistant"
    }
  ],
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "reasoning": {
    "effort": "medium",
    "summary": null
  },
  "service_tier": "default",
  "store": true,
  "temperature": 0.7,
  "text": {
    "format": {
      "type": "text"
    }
  },
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "truncation": "auto",
  "usage": {
    "input_tokens": 13,
    "i

In [ ]:
# Function to convert Azure response format to OpenAI format
def convert_azure_to_openai_format(azure_response):
    """
    Convert Azure OpenAI API response format to the standard OpenAI format
    to maintain compatibility with the rest of the application code.
    
    This function takes a response from Azure OpenAI and restructures it
    to match what the standard OpenAI API would return.
    """
    # Check if there's an error in the Azure response
    if "error" in azure_response and azure_response["error"] is not None:
        return {
            "error": {
                "message": str(azure_response["error"]),
                "type": "azure_api_error"
            }
        }
    
    # For successful responses, maintain the format that the agent expects
    # The main part the agent uses is the 'output' array
    return {
        "id": azure_response.get("id", ""),
        "object": azure_response.get("object", "response"),
        "created": azure_response.get("created_at", 0),
        "model": azure_response.get("model", ""),
        "output": azure_response.get("output", []),
        "usage": azure_response.get("usage", {})
    }

# Test the conversion function with our example response
openai_format_response = convert_azure_to_openai_format(response)
print("Converted to OpenAI format:")
print(json.dumps(openai_format_response, indent=2))

In [ ]:
# Enhanced version with requests - more similar to how utils.py works
import requests

def create_response_azure(**kwargs):
    """
    Function that communicates with Azure OpenAI API and converts the response
    to match the format expected by the OpenAI API.
    
    Parameters are the same as OpenAI's create_response:
    - model: The model ID to use
    - input: The conversation or text input
    - tools: List of tool definitions
    - temperature: Optional temperature parameter
    - truncation: How to handle truncation ("auto" by default)
    """
    # Get Azure-specific environment variables
    azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "https://ai-giotesthub416362359489.openai.azure.com")
    api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2025-03-01-preview")
    
    if not azure_api_key:
        raise ValueError("AZURE_OPENAI_API_KEY environment variable is required")
    
    # Construct the Azure OpenAI API URL
    url = f"{azure_endpoint}/openai/responses?api-version={api_version}"
    
    # Azure uses api-key header instead of Bearer token
    headers = {
        "api-key": azure_api_key,
        "Content-Type": "application/json"
    }
    
    # Send the request
    response = requests.post(url, headers=headers, json=kwargs)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code} {response.text}")
        return {"error": response.text}
    
    # Convert the Azure response to OpenAI format
    azure_response = response.json()
    return convert_azure_to_openai_format(azure_response)

# Example usage with our conversion function
try:
    # Create a simple prompt as a message object
    test_input = [{"role": "user", "content": "Hello, can you tell me about Paris?"}]
    
    # Call our function to send the request to Azure and get response in OpenAI format
    result = create_response_azure(
        model="computer-use-preview",
        input=test_input,
        temperature=0.7,
        truncation="auto"
    )
    
    print("\nResponse from create_response_azure:")
    print(json.dumps(result, indent=2))
except Exception as e:
    print(f"Error testing create_response_azure: {e}")